<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 07
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Walk-Forward Validation &amp; Expected Returns
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Validates a pooled LightGBM baseline with leakage-safe walk-forward folds, measures how much of each prediction survives out of sample, and turns only what survives into the expected returns and volatilities the optimizer consumes.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Key Objectives in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Walk-Forward Baseline:</b> Four chronological folds with a label purge and 20-day embargo; two LightGBM models per fold (<code>Forward_Return_20D</code>, <code>Forward_Volatility_20D</code>), scored with cross-sectional Information Coefficient.</li>
            <li><b>Level &amp; Calibration Checks:</b> Whether return predictions beat a flat mean, and whether predicted volatility is unbiased &mdash; the two properties the optimizer actually depends on.</li>
            <li><b>Return Shrinkage:</b> An out-of-sample calibration slope <i>k</i>, learned fold by fold, decides how much of the model's within-class return spread to keep.</li>
            <li><b>Live Inference:</b> Final models refit on all labelled history predict each ticker's latest row.</li>
            <li><b>Synthetic Assets:</b> Bond and RealEstate use their known generation parameters instead of model forecasts.</li>
            <li><b>Expected Returns:</b> A risk-based prior (equal class Sharpe, within-class beta) plus the gated model tilt, on the same 20-day horizon as the volatility inputs.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Covariance Construction &rarr; NSGA-II Optimization
    </p>
</div>

In [ ]:
#pip install lightgbm

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.config import FEATURE_DIR, DATA_DIR
from src.forecasting.baseline_model import run_walk_forward_baseline
from src.forecasting.baseline_plots import plot_ic_by_fold
from src.forecasting.walk_forward import walk_forward_splits
from src.helpers.charts import chart_dir
from scipy.stats import spearmanr
import lightgbm as lgb

CHART_DIR = chart_dir(DATA_DIR, "walk_forward_baseline")

In [ ]:
df = pd.read_parquet(FEATURE_DIR / "model_ready_dataset.parquet")

feature_cols = [
    "Log_Return_1D", "Log_Return_5D", "Log_Return_20D", "Log_Return_60D",
    "RSI14", "MACD_Histogram", "EMA20_Ratio", "EMA50_Ratio",
    "Volatility_20D", "Volatility_60D", "Bollinger_Width", "Max_Drawdown_60D",
    "Volume_Change", "Relative_Volume", "OBV_Z_Score",
    "Asset_Class",
]
categorical_cols = ["Asset_Class"]
target_cols = ["Forward_Return_20D", "Forward_Volatility_20D"]

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Walk-Forward Baseline: Return &amp; Volatility Models
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Splits the model-ready dataset into four chronological folds, trains one LightGBM model per fold per target on past data only, and predicts the next block &mdash; four folds &times; two targets = eight models. Every evaluation later in this notebook uses only these out-of-sample predictions.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Leakage control:</b> each fold trains on all dates before it (expanding window) and stops 21 trading days before the test block, so no 20-day label overlapping a test date ever reaches training. A random K-fold split would train on the future.</li>
            <li><b><code>Asset_Class</code> is a native categorical feature:</b> one pooled model conditions its splits per class instead of six thin per-class models.</li>
            <li><b>Scored with cross-sectional IC:</b> Spearman correlation between predicted and realized values within each Date &times; Asset_Class group of at least 3 assets, averaged over dates (ICIR = mean / standard deviation). Asset class with 2 assets gets a pairwise hit rate instead; Asset class with 1 asset cannot be ranked and are handled in Step 06.</li>
            <li>The IC chart is saved to <code>../walk_forward_baseline/&lt;run date&gt;/ic_by_fold.png</code>.</li>
        </ul>
    </div>
</div>

In [ ]:
# 1. Unpack both results and predictions
results, oos_preds = run_walk_forward_baseline(
    df, feature_cols=feature_cols, categorical_cols=categorical_cols,
    target_cols=target_cols, n_folds=4, embargo_days=20,
)

display(results)
plot_ic_by_fold(results, save_dir=CHART_DIR)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Baseline Signal Strength &amp; Model Stability
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        The two targets behave very differently: the return model shows <b>no stable cross-sectional ranking skill</b>, while the volatility model shows a strong and persistent signal.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Return IC:</b> 0.018, &minus;0.042, 0.024 and &minus;0.052 across Folds 1&ndash;4 (mean &asymp; &minus;0.013). The sign flips from fold to fold and the ICIR flips with it, so there is no evidence of stable ranking ability.</li>
            <li><b>Volatility IC:</b> 0.325, 0.380, 0.672 and 0.648, positive in every fold, with ICIR of 0.62, 0.75, 1.99 and 1.92 &mdash; stronger and more stable in Folds 3&ndash;4.</li>
            <li><b>Where the volatility skill comes from:</b> largely volatility persistence (trailing <code>Volatility_20D</code> and <code>Volatility_60D</code> are features). That is expected and is exactly what a risk model needs: ranking is confirmed here, and the level is checked in Step 03.</li>
            <li><b>Hit rates (pairwise concordance):</b> volatility shows stronger concordance than return, particularly for Crypto and Stocks in the later folds; 0.5 is a coin flip. Commodity's rate rests on a single pair per date. Bond and RealEstate return <code>NaN</code> because a single asset cannot form a pair &mdash; they are handled in Step 06.</li>
            <li><b>Most important conclusion:</b> this baseline does not justify adding complexity to force return IC higher. The walk-forward evaluation shows that <b>return forecasting is weak, while volatility forecasting is materially stronger</b>.</li>
        </ul>
    </div>
    <div style="background-color: #eff6ff; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb; margin-top: 14px;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">What Happens to the Weak Return Signal</p>
        <p style="margin: 7px 0 0 0; color: #334155; font-size: 14px; line-height: 1.6;">
            A near-zero IC does not make predictions harmless: used at full size, a wide prediction spread adds error. Step 04 therefore measures, fold by fold and using only earlier folds, how much of the return spread survives out of sample (the calibration slope <i>k</i>), and Step 07 uses only that share &mdash; currently none.
        </p>
    </div>
    <p style="margin-top: 14px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next:</b> Step 02 checks whether the return levels beat a flat mean; Step 03 checks whether the volatility level is unbiased.
    </p>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Return Level Check: Model vs Flat Means
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Tests whether the size of the return predictions carries information, by comparing out-of-sample MSE of three forecasts for every asset and date: the raw prediction, the average prediction across all assets that day (<code>Pooled_Mean</code>), and the average within the asset's class that day (<code>Class_Mean</code>).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 13.5px; line-height: 1.6;">
            <li><code>Pooled_Mean</code> keeps only the model's view of the overall market; <code>Class_Mean</code> adds its view of which class will do better. If <code>Class_Mean</code> does not beat <code>Pooled_Mean</code>, the model's class-level differences are noise.</li>
            <li>MSE is in squared return units, so high-volatility assets (Crypto) dominate it: read the pattern across folds, not the absolute values.</li>
        </ul>
    </div>
</div>

In [ ]:
r = oos_preds[oos_preds["Target"] == "Forward_Return_20D"].copy()
r["Realized"] = df.loc[r.index, "Forward_Return_20D"]
r["Pooled_Mean"] = r.groupby("Date")["Raw_Prediction"].transform("mean")
r["Class_Mean"] = r.groupby(["Date", "Asset_Class"])["Raw_Prediction"].transform("mean")

for col in ["Raw_Prediction", "Pooled_Mean", "Class_Mean"]:
    r[f"MSE_{col}"] = (r[col] - r["Realized"]) ** 2

display(r.groupby("Fold")[["MSE_Raw_Prediction", "MSE_Pooled_Mean", "MSE_Class_Mean"]].mean())

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the MSE Table
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        The model's return levels do not survive out of sample, at either the asset or the class level.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Flattening beats the raw prediction in every fold:</b> <code>Pooled_Mean</code> has 7&ndash;27% lower MSE than <code>Raw_Prediction</code> (Fold 1: 0.0151 vs 0.0208).</li>
            <li><b><code>Class_Mean</code> is no better than <code>Pooled_Mean</code>:</b> it wins only Fold 2, by 0.2%, and is 0.8&ndash;3.8% worse in the other three. The model cannot tell which asset class will do better.</li>
            <li><b>Consequence:</b> class-level expected returns come from an explicit risk-based prior in Step 07, not from the model.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Volatility Calibration Check
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Compares the average predicted volatility with the average realized volatility in each fold. The covariance matrix needs the right <i>level</i> of risk, and a high IC (good ranking) does not guarantee it.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Ratio</b> = <code>Mean_Predicted</code> / <code>Mean_Realized</code>: 1.0 is unbiased, above 1 over-states risk, below 1 under-states it.</li>
            <li>A pooled average is dominated by high-volatility classes, so this check says nothing about Bond or RealEstate &mdash; they are handled in Step 06.</li>
        </ul>
    </div>
</div>

In [ ]:
calibration = oos_preds[oos_preds["Target"] == "Forward_Volatility_20D"].groupby("Fold").apply(
    lambda g: pd.Series({
        "Mean_Predicted": g["Raw_Prediction"].mean(),
        "Mean_Realized": df.loc[g.index, "Forward_Volatility_20D"].mean(),
    })
)
calibration["Ratio"] = calibration["Mean_Predicted"] / calibration["Mean_Realized"]
display(calibration)  # Ratio near 1.0 is well-calibrated; systematically off suggests a scale correction before Sigma

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Calibration Ratio
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        No correction is needed: the ratios sit around 1 and swing in both directions.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>This run:</b> 1.04, 0.82, 1.09 and 0.95 (mean &asymp; 0.98). Errors in both directions point to noise, not a systematic bias, so a blanket scale factor would add error rather than remove it.</li>
            <li><b>Fold 2 under-predicted risk by about 18%:</b> realized risk can exceed the forecast in stressed periods, which is worth remembering when reading the optimizer's risk figures.</li>
            <li><b>Revisit</b> if a future run shows ratios consistently on one side of 1.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Return Shrinkage: How Much of the Model's Spread to Keep
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Measures, out of sample, how much of the model's within-class return differences to trust. For each date and class, predictions and realized returns are centered on their class mean, and <i>k</i> is the regression slope of realized on predicted deviations &mdash; the weight that minimizes squared error, expressed in return units.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Learned sequentially:</b> fold <i>f</i> uses only earlier folds' predictions, minus the last 21 days whose labels resolve inside fold <i>f</i>. Fold 1 has no history, so <i>k</i> = 0.</li>
            <li><b>Honest uncertainty:</b> standard errors use Newey&ndash;West (20 lags) because 20-day labels overlap; rows are weighted by 1 / trailing volatility&sup2; and centered returns are winsorized at 1% / 99%.</li>
            <li><b>Gate:</b> <i>k</i> used = max(0, <i>k</i> &minus; SE), so noise cannot switch the tilt on. One-asset classes drop out automatically; Two assets in a class can still contribute.</li>
            <li><b>Why not IC / reference IC:</b> a rank IC has no units, so dividing it by a fixed constant cannot say how large a prediction should be; the slope <i>k</i> can.</li>
        </ul>
    </div>
</div>

In [ ]:
# ── Cell 4: backtest – is there within-class return skill, out of sample? ──
from src.calibration.shrinkage import calibrate_shrinkage_k, expanding_calibrated_tilt, build_expected_returns, _center

HORIZON = 20
GROUPS = ["Date", "Asset_Class"]

ret_oos = oos_preds[oos_preds["Target"] == "Forward_Return_20D"].copy()
ret_oos["Realized"] = df.loc[ret_oos.index, "Forward_Return_20D"]
ret_oos["Volatility_20D"] = df.loc[ret_oos.index, "Volatility_20D"]

tilt, k_table = expanding_calibrated_tilt(df, oos_preds, target="Forward_Return_20D",
                                          weight_col="Volatility_20D", horizon=HORIZON, conservative=True)
display(k_table.round(4))

# Out-of-sample check: centered SSE of (class mean), (raw), (calibrated tilt)
p_c = _center(ret_oos, "Raw_Prediction", GROUPS)
r_c = _center(ret_oos, "Realized", GROUPS)
rows = []
for fold, idx in ret_oos.groupby("Fold").groups.items():
    sse0 = (r_c[idx] ** 2).sum()
    rows.append({"Fold": fold,
                 "Gain_raw_%": 100 * (1 - ((r_c[idx] - p_c[idx]) ** 2).sum() / sse0),
                 "Gain_calibrated_%": 100 * (1 - ((r_c[idx] - tilt[idx]) ** 2).sum() / sse0)})
display(pd.DataFrame(rows).round(3))

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Shrinkage Table
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        The model's within-class return spread does not survive out of sample, so it receives zero weight.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b><i>k</i> is indistinguishable from zero in every fold:</b> <code>k_raw</code> = &minus;0.04, 0.06 and 0.06 with standard errors of 0.09&ndash;0.14 (|t| below 0.7), so <code>k_used</code> = 0 throughout.</li>
            <li><b>Using the raw spread would have hurt:</b> <code>Gain_raw_%</code> is &minus;46, &minus;10, &minus;26 and &minus;24 &mdash; up to 46% more squared error than simply predicting the class mean.</li>
            <li><b>This is the correct outcome, not a failure.</b> A near-zero IC combined with a wide prediction spread destroys value at full size. The gate switches the tilt on automatically if a future model earns it.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 05
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Live Predictions: Final Models
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Refits both models on all labelled history with the same parameters used in the folds, then predicts each ticker's latest row. The model-ready dataset drops the last 20 days because their labels do not exist yet, so live features are read from the post-EDA file.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Guards on the live set:</b> one common date, every ticker present, and <code>Asset_Class</code> categories aligned to the training encoding &mdash; each failure stops the run instead of silently corrupting predictions.</li>
            <li><b>Raw predictions are inputs, not final answers:</b> Steps 06 and 07 decide how much of each is used.</li>
        </ul>
    </div>
</div>

In [ ]:
# ── Cell 5: live predictions – final models on all labelled history ──
import lightgbm as lgb
from src.forecasting.baseline_model import DEFAULT_LGB_PARAMS   # same parameters as the walk-forward folds

X_train = df[feature_cols].copy()
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")

final_models = {}
for target in target_cols:
    labelled = df[target].notna()
    final_models[target] = lgb.LGBMRegressor(**DEFAULT_LGB_PARAMS).fit(
        X_train[labelled], df.loc[labelled, target], categorical_feature=categorical_cols
    )

# model_ready_dataset dropped the last 20 days (no labels yet); their features live in the post-EDA file
live_features = pd.read_parquet(FEATURE_DIR / "clean_features_after_EDA.parquet")
is_latest = live_features["Date"] == live_features.groupby("Ticker")["Date"].transform("max")
df_latest = live_features.loc[is_latest].copy()

latest_dates = df_latest["Date"].unique()
assert len(latest_dates) == 1, f"tickers end on different dates: {sorted(pd.to_datetime(latest_dates).date)}"
missing = sorted(set(df["Ticker"]) - set(df_latest["Ticker"]))
assert not missing, f"no live row for: {missing}"

X_live = df_latest[feature_cols].copy()
for col in categorical_cols:
    # same category order as training, so each class maps to the code the model learned
    X_live[col] = pd.Categorical(X_live[col], categories=X_train[col].cat.categories)
assert X_live[categorical_cols].notna().all().all(), "live data has an asset class the model never saw"

df_latest["Predicted_Return_Raw"] = final_models["Forward_Return_20D"].predict(X_live)
df_latest["Predicted_Volatility_Final"] = final_models["Forward_Volatility_20D"].predict(X_live)

print(f"{len(df_latest)} tickers, live date {pd.Timestamp(latest_dates[0]):%Y-%m-%d}")
display(df_latest[["Ticker", "Asset_Class", "Predicted_Return_Raw", "Predicted_Volatility_Final"]]
        .sort_values(["Asset_Class", "Predicted_Return_Raw"]))


<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        What the Live Set Contains
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        49 tickers on a single live date with a raw 20-day return forecast and an annualized volatility forecast.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Volatility forecasts</b> feed the covariance matrix and the class priors directly; their ranking and level were checked in Steps 01 and 03.</li>
            <li><b>Return forecasts</b> enter only through the gated tilt (<i>k</i> = 0 here), so their raw spread does not reach the optimizer.</li>
        </ul>
    </div>
</div>

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 06
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Bond &amp; RealEstate: Known Parameters
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Bond and RealEstate are synthetic EUR series generated in notebook 03 with Geometric Brownian Motion from fixed inputs (annual return and volatility in <code>portfolio_platform5_input.csv</code>). Their true expected return and volatility are therefore known, so they replace the model's forecasts.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Why they are not evaluated:</b> their daily returns are a constant drift plus independent random shocks (Bond is fully deterministic), so there is nothing to forecast &mdash; any time-series IC would be noise around zero by construction.</li>
            <li><b>Expected return:</b> Bond = its yield (3.5%), which also serves as the risk-free rate; RealEstate = its input annual return (8.5%). Both are pinned in Step 07.</li>
            <li><b>Volatility:</b> full-history realized volatility, constant by construction. Bond: model 0.70% &rarr; known 0.14%, which is not real risk (removing weekends puts three days of accrual into each Monday's return); a small floor keeps it above zero so the optimizer never divides by zero. RealEstate: model 16.27% &rarr; known 12.61%, so the model over-stated its risk by about 29%.</li>
        </ul>
    </div>
</div>   </div>
</div>

In [ ]:
# ── Cell 6: Bond and RealEstate – known parameters, not model forecasts ──
# Both are synthetic GBM series from notebook 03, so their true return and volatility are its inputs.
RF_ANNUAL = 0.035     # <- Bond annual_return / 100 from portfolio_platform5_input.csv
RE_RETURN = 0.085    # <- RealEstate annual_return / 100 from portfolio_platform5_input.csv

for cls in ["Bond", "RealEstate"]:
    r = live_features.loc[live_features["Asset_Class"] == cls, "Log_Return_1D"]
    known_vol = max(r.std() * np.sqrt(252), 1e-4)  # constant by construction; floor avoids a 0/0 Sharpe for Bond
    m = df_latest["Asset_Class"] == cls
    print(f"{cls:<10} model forecast {df_latest.loc[m, 'Predicted_Volatility_Final'].iloc[0]:.2%}  ->  known {known_vol:.2%}")
    df_latest.loc[m, "Predicted_Volatility_Final"] = known_vol

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 07
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Expected Returns for the Optimizer
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Builds each asset's 20-day expected return as a risk-based prior plus the gated model tilt: &mu;<sub>i</sub> = r<sub>f</sub> + &beta;<sub>i</sub> (class prior &minus; r<sub>f</sub>) + <i>k</i> &times; (prediction<sub>i</sub> &minus; class mean).
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 12.5px; line-height: 1.6;">
            <li><b>Class prior:</b> every class's equal-weighted portfolio earns the same Sharpe ratio (SR0 = 0.30). This is an explicit assumption, because Step 02 showed the model cannot set class levels.</li>
            <li><b>&beta;<sub>i</sub>:</b> beta to the equal-weighted class portfolio, from trailing 252-day correlations and the predicted volatilities. Shrunk one-third toward 1 (Blume, 1971), bounded to 0.25&ndash;3.0 and averaging exactly 1 in each class, so class levels are unchanged. Used only where past betas predicted future betas better than &ldquo;all betas = 1&rdquo;.</li>
            <li><b>Pinned classes:</b> Bond at r<sub>f</sub> and RealEstate at its known return (Step 06). The tilt term is zero while <i>k</i> = 0 (Step 04).</li>
            <li><b>Units:</b> every output is on the 20-day horizon &mdash; expected simple return and &sigma;<sub>20D</sub> = &sigma;<sub>annual</sub> &times; &radic;(20/252).</li>
        </ul>
    </div>
</div>

In [ ]:
# ── Cell 7: live expected returns for the optimizer ──
from src.calibration.shrinkage import class_betas, class_portfolio_vol, beta_stability

assert df_latest["Date"].nunique() == 1, "df_latest must hold one date"
assert df_latest["Ticker"].is_unique, "one live row per ticker"

# (a) Model tilt, gated by out-of-sample evidence (k = 0 means the model adds nothing yet)
live = calibrate_shrinkage_k(ret_oos, weight_col="Volatility_20D", horizon=HORIZON)   # all OOS rows with a realised label
K_LIVE = live["k_conservative"]
print(f"k_raw = {live['k_raw']:.3f}  SE = {live['se']:.3f}  t = {live['t_stat']:.2f}  ->  k used = {K_LIVE:.3f}")

# (b) Risk-based spread inside each class: beta to the equal-weighted class portfolio,
#     used only where past betas predicted future betas better than "all betas = 1"
classes = df_latest.set_index("Ticker")["Asset_Class"]
stability = beta_stability(live_features, classes)
display(stability.round(3))
reliable = stability.index[stability["Shrunk_vs_flat"] < 1].tolist()

pred_vol = df_latest.set_index("Ticker")["Predicted_Volatility_Final"]
betas = class_betas(live_features, classes, vol=pred_vol)
betas[~classes.isin(reliable)] = 1.0
print("beta spread used for:", reliable)

# (c) Class level: every class's equal-weighted PORTFOLIO earns Sharpe SR0 (Bond pinned to rf)
class_vol = class_portfolio_vol(live_features, classes, vol=pred_vol)

SR0 = 0.30         # <- equal-Sharpe assumption for class levels
mu = build_expected_returns(df_latest, pred_col="Predicted_Return_Raw", vol_col="Predicted_Volatility_Final",
                            k_tilt=K_LIVE, rf_annual=RF_ANNUAL, sr0=SR0, horizon=HORIZON,
                            overrides={"Bond": RF_ANNUAL, "RealEstate": RE_RETURN}, beta=betas, class_vol=class_vol)
# optimizer inputs on the SAME 20-day horizon:
mu["Predicted_Volatility_20D"] = mu["Predicted_Volatility_Final"] * np.sqrt(HORIZON / 252)
mu["Class_Portfolio_Vol"] = mu["Ticker"].map(class_vol)
check = mu.groupby("Asset_Class").agg(Class_Portfolio_Vol=("Class_Portfolio_Vol", "first"),
                                      Class_Prior_Annual=("Class_Prior_20D", lambda x: x.iloc[0] * 252 / HORIZON),
                                      Mean_Prior_Annual=("Prior_20D", lambda x: x.mean() * 252 / HORIZON))
check["Class_Sharpe"] = (check["Mean_Prior_Annual"] - RF_ANNUAL) / check["Class_Portfolio_Vol"]
display(check.round(4))   # Class_Sharpe = SR0 for every class except the pinned Bond and RealEstate

display(mu[["Ticker", "Asset_Class", "Beta_Class", "Class_Prior_20D", "Prior_20D", "Tilt_20D",
            "Expected_Return_20D", "Predicted_Volatility_20D"]]
        .sort_values(["Asset_Class", "Expected_Return_20D"], ascending=[True, False]))

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Key Takeaway
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Reading the Expected Returns
    </h3>
    <p style="color: #475569; font-size: 14px; margin-bottom: 14px; line-height: 1.5;">
        Every asset now has its own expected return, and each part of it traces either to evidence in this notebook or to a stated assumption.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <ul style="margin: 0; padding-left: 20px; color: #334155; font-size: 14px; line-height: 1.6;">
            <li><b>Model tilt switched off:</b> live <i>k</i> = 0.032 with a standard error of 0.080 (t = 0.40), so <i>k</i> used = 0.</li>
            <li><b>The beta spread is a reliable risk measure:</b> past betas rank future betas with correlation 0.71&ndash;0.81 and cut the error of &ldquo;all betas = 1&rdquo; to 23&ndash;54%, at 89&ndash;100% of test dates, in all four multi-asset classes.</li>
            <li><b>Class levels:</b> each market class has Sharpe 0.30 by construction &mdash; for example Stocks 8.7% a year at 17.3% portfolio volatility, Crypto 17.6% at 47.2%. Bond earns r<sub>f</sub> (3.5%); RealEstate is pinned to its known return (8.5%).</li>
            <li><b>Largest stock prior:</b> a micro-cap at the beta cap of 3.0 (about 243% volatility), at roughly 19% a year. That is genuine for its risk, and the cap stops it from dominating.</li>
            <li><b>Why priors, not raw forecasts:</b> an optimizer amplifies errors in expected returns, and those errors do far more damage than errors in variances (Chopra &amp; Ziemba, 1993).</li>
            <li><b>Beta differences are risk differences, not return forecasts.</b> Empirically, high-beta assets have earned less than CAPM implies, which is one more reason the betas are shrunk toward 1.</li>
        </ul>
    </div>
</div>

In [ ]:
# ── Cell 8: save the optimizer inputs ──
OPT_COLS = ["Date", "Ticker", "Asset_Class", "Expected_Return_20D", "Predicted_Volatility_20D",
            "Predicted_Volatility_Final", "Beta_Class", "Prior_20D", "Tilt_20D"]
optimizer_inputs = (mu[OPT_COLS]
                    .rename(columns={"Predicted_Volatility_Final": "Predicted_Volatility_Annual"})
                    .reset_index(drop=True))
optimizer_inputs["RF_20D"] = RF_ANNUAL * HORIZON / 252   # the same risk-free rate the priors used

assert len(optimizer_inputs) == df_latest["Ticker"].nunique(), "one row per live ticker"
assert optimizer_inputs[["Expected_Return_20D", "Predicted_Volatility_20D"]].notna().all().all(), "missing optimizer input"

out_path = DATA_DIR / "interim" / "optimizer_inputs.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
optimizer_inputs.to_parquet(out_path, index=False)
print(f"saved {len(optimizer_inputs)} rows to {out_path}")